In [1]:
import kagglehub
from pathlib import Path
import shutil
import random
import torch
import numpy as np

# create root dir for data
root = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
out = root / "data" / "conll2003"
out.mkdir(parents=True, exist_ok=True)

# pull data from kaggle
path = kagglehub.dataset_download("juliangarratt/conll2003-dataset")
src = Path(path)
for n in ["eng.train", "eng.testa", "eng.testb"]:
    f = src / n if (src / n).exists() else next(src.rglob(n), None)
    if f:
        shutil.copy(f, out / n)

print("Original CoNLL-2003:", out)

Original CoNLL-2003: /home/arda/Documents/uva_msds/Sem4/DeepLearning/ds6050-project/data/conll2003


In [ ]:
def set_seeds_to(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

set_seeds_to(1234)
torch.backends.cudnn.deterministic = True

In [3]:
def parse_conll(filepath):
    """Read a CoNLL file and return a list of sentences.
    Each sentence is a list of (word, ner_tag) tuples."""
    sentences = []
    current = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            # skip DOCSTART and blank lines
            if line.startswith("-DOCSTART-"):
                continue
            if line == "":
                if current:
                    # end of sentence, add to list
                    sentences.append(current)
                    current = []
            else:
                # columns are: word, POS, chunk, NER
                parts = line.split()
                word = parts[0]
                ner  = parts[-1]
                current.append((word, ner))
        if current:
            sentences.append(current)
    return sentences

train_sentences = parse_conll(out / "eng.train")
dev_sentences   = parse_conll(out / "eng.testa")
test_sentences  = parse_conll(out / "eng.testb")

print(f"Train: {len(train_sentences)} sentences")
print(f"Dev:   {len(dev_sentences)} sentences")
print(f"Test:  {len(test_sentences)} sentences")

Train: 14041 sentences
Dev:   3250 sentences
Test:  3453 sentences


Note:
- Train: 14,987 sentences - 946 (DOCSTART_Count) = 14,041 without -DOCSTART-
- We need a `parse_conll_with_docs` function for the doc context models

In [4]:
set(tag for sentence in train_sentences for word, tag in sentence)

{'B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O'}

In [5]:
label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", 
              "B-LOC", "I-LOC", "B-MISC", "I-MISC"]

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

In [6]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    get_linear_schedule_with_warmup
)
from seqeval.metrics import classification_report

In [7]:
MODEL_NAME = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NERDataset(Dataset):
    def __init__(self, sentences, tokenizer, label2id, max_length=512):
        self.sentences  = sentences
        self.tokenizer  = tokenizer
        self.label2id   = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        words = [w for w, _ in sentence]
        tags  = [t for _, t in sentence]
        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors="pt"
        )
        word_ids = encoding.word_ids()
        labels = []
        for i, word_id in enumerate(word_ids):
            if word_id is None:
                labels.append(-100)
            elif i == 0 or word_ids[i] != word_ids[i - 1]:
                labels.append(self.label2id[tags[word_id]])
            else:
                labels.append(-100)
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         torch.tensor(labels, dtype=torch.long)
        }

def collate_fn(batch):
    """Pad to max len in batch — assignment 3 style."""
    max_len = max(item["input_ids"].size(0) for item in batch)
    input_ids_list, attention_mask_list, labels_list = [], [], []
    for item in batch:
        pad_len = max_len - item["input_ids"].size(0)
        input_ids_list.append(
            torch.nn.functional.pad(item["input_ids"], (0, pad_len), value=0)
        )
        attention_mask_list.append(
            torch.nn.functional.pad(item["attention_mask"], (0, pad_len), value=0)
        )
        labels_list.append(
            torch.nn.functional.pad(item["labels"], (0, pad_len), value=-100)
        )
    return (
        torch.stack(input_ids_list),
        torch.stack(attention_mask_list),
        torch.stack(labels_list)
    )

train_dataset = NERDataset(train_sentences, tokenizer, label2id)
dev_dataset   = NERDataset(dev_sentences,   tokenizer, label2id)
test_dataset  = NERDataset(test_sentences,  tokenizer, label2id)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,  collate_fn=collate_fn)
dev_loader   = DataLoader(dev_dataset,   batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}")
print(f"Dev batches:   {len(dev_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 878
Dev batches:   204
Test batches:  216


In [8]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Using device: {device}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Using device: cuda
Number of parameters: 107,726,601


In [9]:
epochs    = 5
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=len(train_loader) * epochs
)

for epoch in range(epochs):
    # training
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids, attention_mask, labels = batch
        optimizer.zero_grad()
        outputs = model(
            input_ids      = input_ids.to(device),
            attention_mask = attention_mask.to(device),
            labels         = labels.to(device)
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # validation F1 after each epoch
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for batch in dev_loader:
            input_ids, attention_mask, labels = batch
            outputs = model(
                input_ids      = input_ids.to(device),
                attention_mask = attention_mask.to(device)
            )
            predictions = torch.argmax(outputs.logits, dim=-1)
            for pred_seq, label_seq in zip(predictions, labels):
                pred_tags, true_tags = [], []
                for p, l in zip(pred_seq, label_seq):
                    if l == -100:
                        continue
                    pred_tags.append(id2label[p.item()])
                    true_tags.append(id2label[l.item()])
                all_preds.append(pred_tags)
                all_true.append(true_tags)

    report = classification_report(all_true, all_preds, output_dict=True)
    dev_f1  = report["micro avg"]["f1-score"]
    print(f"Epoch {epoch+1}/{epochs} — Loss: {avg_loss:.4f} — Dev F1: {dev_f1:.4f}")

Epoch 1/5 — Loss: 0.1124 — Dev F1: 0.9386
Epoch 2/5 — Loss: 0.0279 — Dev F1: 0.9490
Epoch 3/5 — Loss: 0.0145 — Dev F1: 0.9488
Epoch 4/5 — Loss: 0.0081 — Dev F1: 0.9481
Epoch 5/5 — Loss: 0.0054 — Dev F1: 0.9510


## Test Set Evaluation

Results are reported using entity-level F1 via `seqeval`.

The table below shows precision, recall, and F1 broken down by:
- **Per entity type** (PER, ORG, LOC, MISC) — how well the model performs on each category
- **micro avg** — F1 computed globally across every entity, treating all instances equally regardless of type.
- **macro avg** — Unweighted average of F1 across the four entity types, giving equal weight to each type regardless of frequency
- **weighted avg** — aAerage of F1 across entity types weighted by how many instances of each type appear in the test set

In [15]:
import pandas as pd

model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = batch
        outputs = model(
            input_ids      = input_ids.to(device),
            attention_mask = attention_mask.to(device)
        )
        predictions = torch.argmax(outputs.logits, dim=-1)
        for pred_seq, label_seq in zip(predictions, labels):
            pred_tags, true_tags = [], []
            for p, l in zip(pred_seq, label_seq):
                if l == -100:
                    continue
                pred_tags.append(id2label[p.item()])
                true_tags.append(id2label[l.item()])
            all_preds.append(pred_tags)
            all_true.append(true_tags)

report = classification_report(all_true, all_preds, output_dict=True)
df = pd.DataFrame(report).T.round(4)

print("=== Test Set Evaluation (detailed) ===")
display(df)

=== Test Set Evaluation (detailed) ===


,precision,recall,f1-score,support
LOC,0.9349,0.9382,0.9366,1668.0
MISC,0.7832,0.8234,0.8028,702.0
ORG,0.8724,0.9097,0.8907,1661.0
PER,0.9661,0.9530,0.9595,1617.0
micro avg,0.9052,0.9198,0.9124,5648.0
macro avg,0.8892,0.9061,0.8974,5648.0
weighted avg,0.9066,0.9198,0.9130,5648.0


In [14]:
from seqeval.metrics import f1_score, precision_score, recall_score

print("=== Test Set Evaluation (summary) ===") 
print(f"Precision : {precision_score(all_true, all_preds):.4f}")
print(f"Recall    : {recall_score(all_true, all_preds):.4f}")
print(f"F1        : {f1_score(all_true, all_preds):.4f}")

=== Test Set Evaluation (summary) ===
Precision : 0.9052
Recall    : 0.9198
F1        : 0.9124
